Import Necessary Libraries

In [9]:
import sys
from pathlib import Path

# We are inside src/pipeline → go up one level to src
src_path = Path().resolve().parents[0]

sys.path.insert(0, str(src_path))

print("SRC path added:", src_path)

SRC path added: D:\Workspace\PySpark\src


In [10]:
import bronze
print("Bronze imported successfully 🎉")

Bronze imported successfully 🎉


In [ ]:
import uuid
from bronze.audit import *
from bronze.config import *
from bronze.extractor import *
from bronze.file_tracker import *
from bronze.incremental_loader import *
from bronze.log_metrics import *
from bronze.logger import *
from bronze.quality import *
from bronze.reader import *
from bronze.transform import *
from bronze.writer import *
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StructType, IntegerType, DoubleType, DateType, LongType, StringType
from pyspark.sql import functions as f
from datetime import datetime

import sys



ModuleNotFoundError: No module named 'file_tracker'

Configure Spark Sessiona and Delta-Spark

In [2]:
spark = (
    SparkSession.builder
    .appName("Retail Transformation1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [3]:
def create_audit_table(spark, audit_path):
    spark.sql(f"""
              CREATE TABLE IF NOT EXISTS delta.`{audit_path}`
              (
                  pipeline_name STRING,
                  batch_id STRING,
                  mode STRING,
                  status STRING,
                  start_time TIMESTAMP,
                  end_time TIMESTAMP,
                  duration_seconds DOUBLE,
                  created_at TIMESTAMP
              )
              USING DELTA
            """)

In [4]:
def log_pipeline_run(spark, audit_path, run_metadata):
    
    audit_df = spark.createDataFrame([run_metadata]) \
                    .withColumn("created_at", f.current_timestamp())
    
    audit_df.write \
            .format("delta") \
            .mode("append") \
            .save(audit_path)

In [5]:

raw_schema = StructType(fields = [StructField("order_id", StringType()),
                                  StructField("customer_id", StringType()),
                                  StructField("customer_name", StringType()),
                                  StructField("email", StringType()),
                                  StructField("phone", LongType()),
                                  StructField("product_id", StringType()),
                                  StructField("product_name", StringType()),
                                  StructField("category", StringType()),
                                  StructField("quantity", IntegerType()),
                                  StructField("price", DoubleType()),
                                  StructField("discount", DoubleType()),
                                  StructField("order_date", DateType()),
                                  StructField("shipment_date", DateType()),
                                  StructField("payment_method", StringType()),
                                  StructField("store_location", StringType()),
                                  StructField("status", StringType())])


In [6]:
def run_pipeline(mode="batch"):
    config = EnvConfig()
    logger = get_logger()
    
    pipeline_name = "bronze_retail"
    batch_id = str(uuid.uuid4())
    
    start_time = datetime.now()
    status = "Running"
    
    create_audit_table(spark, config.audit_path)
    
    try:
        logger.info("=====================================================================")
        logger.info("Bronze Pipeline Started")
        logger.info(f"Pipeline Name : {pipeline_name}")
        logger.info(f"Batch_Id : {batch_id}")
        logger.info(f"Mode : {mode}")
        logger.info(f"Environment : {config.env}")
        logger.info("=====================================================================")
    
        
        schema = raw_schema
        
    
        #Batch Mode
        if mode == "batch":
            logger.info("Detecting new files to ingest")
            
            new_files = get_new_files(spark, config, logger)
            
            logger.info(f"New files detected: {len(new_files)}")
            
            if len(new_files) == 0:
                logger.info("No New Files found. Pipeline exiting")
                return
            else:
                
                logger.info("Reading new batch files")
                logger.info(f"New files list raw object: {new_files}")
                logger.info(f"Type of first item: {type(new_files[0])}")
                df = read_batch(spark, new_files, config, schema)
            
                logger.info("Applying Bronze Transformation")
                df = bronze_transform(df, batch_id)
            
                logger.info("Writing to Bronze Layer")
                write_bronze_batch(df, config, logger, batch_id)
            
            
        
        #Stream Mode
        elif mode == "stream":
        
            logger.info("Starting Streaming Ingestion")
                        
            df = read_stream(spark,config,schema)
            df = bronze_transform(df, batch_id)
            
            query = write_bronze_stream(df, config, logger)
            logger.info("Streaming job started successfully")
            query.awaitTermination()
            
        else:
            raise ValueError("Mode must be either 'Batch' or 'Stream'")
        
        status = "SUCCESS"
        
    #Error Handling
    except Exception as e:
        status = "FAILED"
        logger.exception("Pipeline FAILED with error")
        raise
    
    #Pipeline End
    finally:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        # Derive final status safely
    
        final_status = "FAILED" if sys.exc_info()[0] else "SUCCESS"

        run_metadata = {
            "pipeline_name": pipeline_name,
            "batch_id": batch_id,
            "mode": mode,
            "status": final_status,
            "start_time": start_time,
            "end_time": end_time,
            "duration_seconds": duration
        }

        log_pipeline_run(spark, config.audit_path, run_metadata)

        logger.info("=====================================================================")
        logger.info(f"Bronze Pipeline Finished with status: {final_status}")
        logger.info(f"Duration: {duration} seconds")
        logger.info("=====================================================================")
            

In [7]:
run_pipeline("batch")

2026-04-02 00:55:40,536 | INFO | =====================================================================
2026-04-02 00:55:40,536 | INFO | Bronze Pipeline Started
2026-04-02 00:55:40,536 | INFO | Pipeline Name : bronze_retail
2026-04-02 00:55:40,536 | INFO | Batch_Id : c09b202d-3b3e-4eb7-8626-17178f516187
2026-04-02 00:55:40,536 | INFO | Mode : batch
2026-04-02 00:55:40,536 | INFO | Environment : dev
2026-04-02 00:55:40,536 | INFO | =====================================================================
2026-04-02 00:55:40,536 | INFO | Detecting new files to ingest
2026-04-02 00:55:40,536 | INFO | Scanning raw folder for CSV files
2026-04-02 00:55:43,238 | INFO | Total files found in raw: 2
2026-04-02 00:55:56,995 | INFO | Already processed files: 2
2026-04-02 00:55:56,995 | INFO | New Files detected: 0
2026-04-02 00:55:56,995 | INFO | New files detected: 0
2026-04-02 00:55:57,011 | INFO | No New Files found. Pipeline exiting
2026-04-02 00:56:16,253 | INFO | ================================

In [8]:
config = EnvConfig()

spark.read.format("delta").load(config.audit_path).show(truncate = False)

+-------------+------------------------------------+-----+-------+--------------------------+--------------------------+----------------+--------------------------+
|pipeline_name|batch_id                            |mode |status |start_time                |end_time                  |duration_seconds|created_at                |
+-------------+------------------------------------+-----+-------+--------------------------+--------------------------+----------------+--------------------------+
|bronze_retail|d18c5c88-6c1d-43d8-8d99-e2ee8b45be5e|batch|SUCCESS|2026-03-31 09:20:22.595861|2026-03-31 09:20:37.575049|14.979188       |2026-03-31 09:20:38.086741|
|bronze_retail|7936cc1f-6886-499a-947c-15a9fb1cbff6|batch|SUCCESS|2026-03-31 08:55:49.007633|2026-03-31 08:56:07.832517|18.824884       |2026-03-31 08:56:08.433247|
|bronze_retail|7059ed2c-55e8-4ef1-ab16-bf3105094797|batch|SUCCESS|2026-03-28 19:33:23.198042|2026-03-28 19:33:23.949105|0.751063        |2026-03-28 19:33:24.047052|
|bronze_re

In [9]:
config = EnvConfig()

spark.read.format("delta").load(config.tracker_path).show(truncate = False)

+------------------------------------------------------------------------------+--------------------------+------------------------------------+
|source_file_name                                                              |ingestion_time            |batch_id                            |
+------------------------------------------------------------------------------+--------------------------+------------------------------------+
|file:///D:/Pyspark%20Dataset/data_lake_local/raw/retail1/retail_dataset_v2.csv|2026-04-01 09:24:02.963076|1ff0b091-ccec-48f1-9a14-95975da7af4e|
|file:///D:/Pyspark%20Dataset/data_lake_local/raw/retail1/retail_dataset1.csv  |2026-03-28 19:32:58.102566|d7fea725-ebf6-4d79-be91-41f35fa72f5b|
+------------------------------------------------------------------------------+--------------------------+------------------------------------+



In [10]:
bronze_df_temp = (
        spark.read \
              .format("delta") \
              .load(config.bronze_path)
        )

In [11]:
bronze_df_temp.show(2)

+---------+-----------+-------------+----------------+-----+----------+------------+---------+--------+------+--------+----------+-------------+--------------+--------------+---------+--------------------+--------------------+--------------+--------------------+
| order_id|customer_id|customer_name|           email|phone|product_id|product_name| category|quantity| price|discount|order_date|shipment_date|payment_method|store_location|   status|    source_file_name| ingestion_timestamp|ingestion_date|            batch_id|
+---------+-----------+-------------+----------------+-----+----------+------------+---------+--------+------+--------+----------+-------------+--------------+--------------+---------+--------------------+--------------------+--------------+--------------------+
|ORD137980|   CUST4069|         NULL|user480@mail.com| NULL|   PROD662|        NULL|Furniture|    NULL|194.42|   10.33|2023-09-25|   2023-06-12|           UPI|          NULL|Completed|file:///D:/Pyspar...|2026-0

In [12]:
from silver_order_write import merge_silver_orders, write_silver_quarantine
from extractor import read_bronze_orders
from deduplicatory import deduplicate_orders
from transformer import transform_orders
from logger import get_logger

In [13]:
def run_silver_pipeline(spark, config, logger):
    
    logger.info("Starting Silver Pipeline")
    
    bronze_df = read_bronze_orders(spark, config, logger)
    
    dedup_df = deduplicate_orders(bronze_df, logger)
    
    valid_df, invalid_df = transform_orders(dedup_df, logger)
    
    #Incremental upsert
    
    merge_silver_orders(spark, valid_df, config, logger)
    
    write_silver_quarantine(invalid_df, config, logger)
    
    logger.info("Silver Pipeline completed successfully") 
    


In [14]:
logger = get_logger()
    
run_silver_pipeline(spark, config, logger)
    

2026-04-02 00:56:22,604 | INFO | Starting Silver Pipeline
2026-04-02 00:56:22,606 | INFO | Reading Bronze Retail Table
2026-04-02 00:56:23,523 | INFO | Bronze Row Reads: 52150
2026-04-02 00:56:23,523 | INFO | Deduplicating Orders
2026-04-02 00:56:27,686 | INFO | Rows after Deduplication: 50150
2026-04-02 00:56:27,686 | INFO | Silver Transformations - Applying Standarization
2026-04-02 00:56:28,161 | INFO | Silver Transformation - Applying Data Qualtiy Rules
2026-04-02 00:56:28,181 | INFO | Silver Transformations - Applying Business Rules
2026-04-02 00:56:28,220 | INFO | Silver Transformation - Applying regex
2026-04-02 00:56:34,816 | INFO | Valid Rows: 150
2026-04-02 00:57:40,390 | INFO | Invalid Rows: 50000
2026-04-02 00:57:40,406 | INFO | Starting merge into silver orders
2026-04-02 00:57:53,040 | INFO | Merge completed successfully
2026-04-02 00:57:53,043 | INFO | Writing Silver Quarantine records
2026-04-02 00:58:06,059 | INFO | Silver Pipeline completed successfully


In [15]:
from date_utils import get_date_window, generate_dates_between
from delta.tables import DeltaTable
from datetime import  timedelta

In [22]:
def run_dim_date(spark, config, logger):
    
    logger.info("Starting DIM DATE incremental")
    
    target_path = config.gold_path + "/dim_date"
    
    start_date, end_date = get_date_window()
    
    #Check if DIM DATE exixts    
    if DeltaTable.isDeltaTable(spark, target_path):
        
        logger.info("DIM DATE alreay exist -> checking if extension needed")
        
        existing_df = spark.read.format("delta").load(target_path)
        
        max_existing_date = existing_df.agg(f.max("date")).collect()[0][0]
        
        logger.info(f"Max Existing Date in table: {max_existing_date}")
        logger.info(f"Pipeline end date window : {end_date}")

        end_date_dt = datetime.strptime(end_date, "%Y-%m-%d").date()
        
        if max_existing_date >= end_date_dt:
            logger.info("DIM DATE already up to date. No new dates required")
            return
            
        #Generate only needed for missing future dates
        
        new_start_date = max_existing_date + timedelta(days = 1)
        
        logger.info(f"Generating new dates from {new_start_date} to {end_date}")
        
        new_dates_df = generate_dates_between(spark, str(new_start_date),  end_date)
        
        new_dates_df.write.format("delta").mode("append").save(target_path)
            
        logger.info("DIM DATE extended successfully")
            
    else:
        
        logger.info("DIM DATE does not exists. creating new table")
        
        dime_date_df = generate_dates_between(spark, start_date, end_date)
        
        dime_date_df.write.format("delta").mode("overwrite").save(target_path)
        
        logger.info("DIM DATE created successfully")       
        
           

In [23]:
logger = get_logger()

run_dim_date(spark, config, logger)

2026-04-02 01:17:13,844 | INFO | Starting DIM DATE incremental
2026-04-02 01:17:13,912 | INFO | DIM DATE alreay exist -> checking if extension needed
2026-04-02 01:17:16,393 | INFO | Max Existing Date in table: 2036-12-31
2026-04-02 01:17:16,399 | INFO | Pipeline end date window : 2036-12-31
2026-04-02 01:17:16,399 | INFO | DIM DATE already up to date. No new dates required
